<a href="https://colab.research.google.com/github/CyberNelson/Python-Tutor/blob/main/Scraping_NFL_Football_Statistics_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas requests lxml selenium
!apt-get update
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

     |████████████████████████████████| 954 kB 5.2 MB/s 
     |████████████████████████████████| 138 kB 43.8 MB/s 
     |████████████████████████████████| 356 kB 48.5 MB/s 
     |████████████████████████████████| 138 kB 48.7 MB/s 
     |████████████████████████████████| 138 kB 47.0 MB/s 
     |████████████████████████████████| 153 kB 62.4 MB/s 
     |████████████████████████████████| 137 kB 53.0 MB/s 
     |████████████████████████████████| 136 kB 34.7 MB/s 
     |████████████████████████████████| 136 kB 47.2 MB/s 
     |████████████████████████████████| 136 kB 48.2 MB/s 
INFO: pip is looking at multiple versions of trio-websocket to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of attrs to determine which version is compatible with other requirements. This could take a while.
     |████████████████████████████████| 53 kB 2.1 MB/s 
     |████████████████████████████████| 49 kB 4.9 MB/s 
INFO: pip is loo

In [ ]:
import pandas as pd 
import lxml.html
from lxml import etree
import sys,datetime,json,time
from selenium import webdriver
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
wd = webdriver.Chrome('chromedriver',chrome_options=chrome_options)
output=None
for year in range(1970,datetime.date.today().year+1):
  print("Extracting Data For Year: ",year)
  pagination_url="https://www.nfl.com/stats/player-stats/category/passing/"+str(year)+"/REG/all/passingyards/desc"
  year_output,page_number=None,1
  while True:
    print("Page Number: ",page_number)
    print("Current Pagination URL: ",pagination_url)
    wait_time=3
    while True:
      try:
        wd.get(pagination_url)
        page=wd.page_source
        doc = lxml.html.fromstring(page)
        table=etree.tostring(doc.xpath("//table[@class='d3-o-table d3-o-table--detailed d3-o-player-stats--detailed d3-o-table--sortable']")[0],pretty_print=True)
        print(pd.read_html(table)[0])
        year_output=pd.read_html(table)[0] if year_output is None else pd.concat([year_output,pd.read_html(table)[0]])
        print("Records Total: ",len(year_output))
        del table
        break
      except Exception as e:
        print("Exception: ",e)
        if len(doc.xpath('//div[@class="nfl-o-table-pagination__buttons"]/a/@href')) is 0 and "Rushing" in page:
          print("Cancelling Page Extract. No Data")
          break
        else:
          print("Next Attempt Begins In",wait_time,"Seconds")
          time.sleep(wait_time)
          wait_time+=wait_time+1
          del page,doc
    try:
      pagination_url="https://www.nfl.com"+doc.xpath('//div[@class="nfl-o-table-pagination__buttons"]/a/@href')[0]
      print("Next Pagination URL: ",pagination_url)
      del page,doc
      page_number+=1
    except Exception as e:
      print("Exception: ",e)
      print("Next Pagination URL: END OF DATASET FOR YEAR ",year)
      break
  del pagination_url,wait_time
  year_output["NFL Year"]=year
  output=year_output if output is None else pd.concat([output,year_output])
  del year_output
print("Player Stats Table Exporting...")
output.to_excel("/content/drive/MyDrive/Fantasy Football/Foundation/NFL Player Stats.xlsx",index=False)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:11: DeprecationWarning: use options instead of chrome_options
  # This is added back by InteractiveShellApp.init_path()


Streaming output truncated to the last 5000 lines.
19       Mike Kirkland       211      5.1   41   19  ...    0    0   34   11    78
20          Randy Dean       188      4.8   39   19  ...    0    0   48    2    16
21       Craig Penrose       185      5.0   37   16  ...    0    0   29    0     0
22            Tom Owen       182      7.0   26   15  ...    0    0   23    3    22
23           Gary Huff       169      4.7   36   15  ...    0    0   31    5    54
24  Steve Pisarkiewicz       164      5.7   29   10  ...    0    0   40    2    20

[25 rows x 16 columns]
Records Total:  50
Next Pagination URL:  https://www.nfl.com/stats/player-stats/category/passing/1978/REG/all/passingYards/DESC?aftercursor=0000003200000000008500100079000840648000000000006e00000005000000045f74626c00000010706572736f6e5f7465616d5f737461740000000565736249640000000950495335353337323200000004726f6c6500000003504c5900000008736561736f6e496400000004313937380000000a736561736f6e5479706500000003524547f07fffffcdf07ffff